# Study RAG pipeline
1. Config  →  2. Chapter metadata  →  3. OCR (cached)  →  4. Pages → chapter documents  →  5. Embedding store (cached)  →  6. Build  →  7. Ask

Run 1–5 every time. Run 6 only when you add books. Run 7 to ask questions.

In [1]:
import os
# Hugging Face Hub cache on Windows is incomplete here (weight file never finished).
# Keep downloads enabled until model_cache has a full snapshot.
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

import json
import re
from collections import defaultdict
from functools import lru_cache
from pathlib import Path

import easyocr
import numpy as np
import torch
import torchvision.transforms.functional as F
from dotenv import load_dotenv
from huggingface_hub import snapshot_download
from pdf2image import convert_from_path
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

ROOT = Path(r"D:\AI-ML\AI Projects\Study RAG")
PDF_DIR = ROOT / "pdf"                   # pdf/Class_6/BGS.pdf
OCR_CACHE = ROOT / "ocr_cache"           # one json per book (raw OCR text)
EMBED_CACHE = ROOT / "embedding_cache"   # class / book / chapter /
MODEL_CACHE = ROOT / "model_cache"
POPPLER_PATH = r"C:\poppler\Library\bin"
KEYWORDS = ("সূচিপত্র", "সুচিপত্র", "বিষয়সূচি", "CONTENTS", "contents")

OCR_CACHE.mkdir(exist_ok=True)
EMBED_CACHE.mkdir(exist_ok=True)
MODEL_CACHE.mkdir(exist_ok=True)
INDEX_INFOS = json.loads((ROOT / "chapters.json").read_text(encoding="utf-8"))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=350,
    separators=["\n\n\n\n", "\n\n\n", "\n\n", "\n", "।", " ", ""],
)

c:\Users\Tanvir\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Tanvir\AppData\Local\Temp\ipykernel_8232\2432406355.py:25: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


## 2. Chapter metadata (from chapters.json)

In [2]:
def get_page_metadatas(class_num, subject, page):
    metas = {}
    if class_num is None or subject is None or page is None or page < 0:
        return metas

    book_section = INDEX_INFOS.get("all_books", {}).get(class_num, {}).get(subject)
    if not book_section:
        return metas

    for chapter in book_section:
        start = chapter.get("starting_page")
        if start is None or start > page:
            continue

        if chapter.get("sub_chapters"):
            for sub in chapter["sub_chapters"]:
                sub_start = sub.get("starting_page")
                if sub_start is not None and sub_start <= page:
                    metas["part"] = chapter.get("chapter_name")
                    metas["chapter"] = sub.get("sub_chapter_name")
                    if sub.get("writer_name"):
                        metas["writer_name"] = sub.get("writer_name")
                    else:
                        metas.pop("writer_name", None)
        else:
            metas["chapter"] = chapter.get("chapter_name")
            metas.pop("part", None)
            metas.pop("writer_name", None)

    return metas

## 3. OCR (GPU, cached — a finished book is never OCR'd again)

In [3]:
_reader = None

def get_reader():
    """Load EasyOCR only when a book actually needs OCR."""
    global _reader
    if _reader is None:
        _reader = easyocr.Reader(["bn", "en"], gpu=torch.cuda.is_available())
    return _reader


def ocr_pdf_to_pages_gpu(pdf_path: Path, poppler_path=POPPLER_PATH, save_every=10):
    class_number = pdf_path.parent.name.lower()
    sub_name = pdf_path.stem.lower()

    cache_file = OCR_CACHE / f"{class_number}_{sub_name}.json"
    partial_file = cache_file.with_suffix(".partial.json")

    if cache_file.exists():
        print(f"OCR loaded from cache: {cache_file.name}")
        texts = json.loads(cache_file.read_text(encoding="utf-8"))
    else:
        # resume a crashed run if a partial file exists
        texts = json.loads(partial_file.read_text(encoding="utf-8")) if partial_file.exists() else []
        if texts:
            print(f"Resuming from page {len(texts) + 1}")

        images = convert_from_path(pdf_path, dpi=300, poppler_path=poppler_path)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        for i, img in tqdm(enumerate(images), total=len(images), initial=len(texts),
                           desc=f"OCR {sub_name} via {device}"):
            if i < len(texts):
                continue  # already done

            img_tensor = F.to_tensor(img).to(device)
            if img_tensor.shape[0] == 3:  # RGB -> grayscale
                img_tensor = (0.2989 * img_tensor[0] + 0.5870 * img_tensor[1]
                              + 0.1140 * img_tensor[2]).unsqueeze(0)

            min_val, max_val = img_tensor.min(), img_tensor.max()
            if max_val > min_val:  # autocontrast
                img_tensor = (img_tensor - min_val) / (max_val - min_val)

            img_numpy = (img_tensor.squeeze(0).cpu().numpy() * 255).astype("uint8")
            results = get_reader().readtext(img_numpy, paragraph=True)
            texts.append("\n".join(r[1] for r in results))

            if len(texts) % save_every == 0:
                partial_file.write_text(json.dumps(texts, ensure_ascii=False), encoding="utf-8")

        cache_file.write_text(json.dumps(texts, ensure_ascii=False), encoding="utf-8")
        partial_file.unlink(missing_ok=True)

    # page numbering starts at the table-of-contents page; earlier pages stay 0
    pages, tracker, page_num = [], False, 0
    for text in texts:
        if any(k in text for k in KEYWORDS):
            tracker = True
        pages.append({
            "page_number": page_num if tracker else 0,
            "page_content": text,
            "book_name": sub_name,
            "class_number": class_number,
        })
        if tracker:
            page_num += 1
    return pages

## 4. Pages → one document per chapter

In [4]:
def pages_to_documents(book_dicts):
    documents = []
    for item in book_dicts:
        content = item.pop("page_content", "")
        metas = get_page_metadatas(item["class_number"], item["book_name"], item["page_number"])
        documents.append(Document(page_content=content, metadata=item | metas))
    return documents


def merge_documents_by_chapter(documents):
    # key includes class + book, so same-named chapters in different books never mix
    grouped = defaultdict(list)
    for doc in documents:
        m = doc.metadata
        grouped[(m["class_number"], m["book_name"], m.get("chapter", "বই তথ্য"))].append(doc)

    merged = []
    for (cls, book, chapter), docs in grouped.items():
        docs = sorted(docs, key=lambda d: d.metadata["page_number"])
        pages = [d.metadata["page_number"] for d in docs]
        merged.append(Document(
            page_content="\n\n".join(d.page_content for d in docs),
            metadata={"class_number": cls, "book_name": book, "chapter": chapter,
                      "page_start": min(pages), "page_end": max(pages)},
        ))
    return merged

## 5. Embedding store (cached as `embedding_cache/class/book/chapter/`)

This cell is self-contained, so you can move it into `embedding_store.py` unchanged and `import` it from LangGraph.

In [5]:
@lru_cache(maxsize=1)
def get_model():
    """Load E5 from the project cache and make the selected device explicit."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading multilingual-e5-large on {device} ...")
    return SentenceTransformer("intfloat/multilingual-e5-large", cache_folder=str(MODEL_CACHE), device=device)


class E5Embeddings(Embeddings):
    """Adds the E5 'query: ' / 'passage: ' prefixes, so the stored text stays clean."""
    def embed_documents(self, texts):
        batch_size = 16 if torch.cuda.is_available() else 4
        return get_model().encode(["passage: " + t for t in texts],
                                  normalize_embeddings=True, batch_size=batch_size,
                                  show_progress_bar=True).tolist()

    def embed_query(self, text):
        return get_model().encode("query: " + text, normalize_embeddings=True).tolist()


def _path(cls, book, chapter):
    chapter = "".join("_" if c in '<>:"/\\|?*' else c for c in chapter).strip().rstrip(".")
    return EMBED_CACHE / cls / book / chapter


def make_embeddings(chunks, cls, book, chapter, force=False):
    """Embed one chapter's chunks and cache them. Skips chapters already cached."""
    out = _path(cls, book, chapter)
    if (out / "vectors.npy").exists() and not force:
        return
    out.mkdir(parents=True, exist_ok=True)
    vecs = np.array(E5Embeddings().embed_documents([c.page_content for c in chunks]), dtype="float32")
    np.save(out / "vectors.npy", vecs)
    (out / "chunks.json").write_text(
        json.dumps([{"text": c.page_content, "meta": c.metadata} for c in chunks],
                   ensure_ascii=False), encoding="utf-8")


def get_embeddings(cls, book, chapter):
    out = _path(cls, book, chapter)
    vecs = np.load(out / "vectors.npy")
    chunks = json.loads((out / "chunks.json").read_text(encoding="utf-8"))
    return vecs, chunks


def list_classes():
    return sorted(p.name for p in EMBED_CACHE.iterdir() if p.is_dir())

def list_books(cls):
    return sorted(p.name for p in (EMBED_CACHE / cls).iterdir() if p.is_dir())

def list_chapters(cls, book):
    return sorted(p.name for p in (EMBED_CACHE / cls / book).iterdir() if p.is_dir())


def _bn_tokenize(text):
    return re.sub(r"[।,;:!?\"'()\[\]{}\-–—.]", " ", text).split()


@lru_cache(maxsize=32)  # built once per chapter, reused on every query
def get_hybrid_retriever(cls, book, chapter, k=5, weights=(0.4, 0.6)):
    vecs, raw = get_embeddings(cls, book, chapter)
    docs = [Document(page_content=c["text"], metadata=c["meta"]) for c in raw]

    # dense: FAISS built from the cached vectors (nothing is re-embedded)
    vectorstore = FAISS.from_embeddings(
        text_embeddings=list(zip([d.page_content for d in docs], vecs)),
        embedding=E5Embeddings(),
        metadatas=[d.metadata for d in docs],
    )
    dense = vectorstore.as_retriever(search_kwargs={"k": k})

    bm25 = BM25Retriever.from_documents(docs, preprocess_func=_bn_tokenize)
    bm25.k = k

    return EnsembleRetriever(retrievers=[bm25, dense], weights=list(weights))

In [6]:
print([p.name for p in MODEL_CACHE.iterdir()])
print((Path.home() / ".cache" / "huggingface" / "hub" / "models--intfloat--multilingual-e5-large").exists())

['.locks', 'models--intfloat--multilingual-e5-large']
True


In [7]:
from pathlib import Path
from huggingface_hub import snapshot_download

# Run this once if get_model reports a missing/incomplete model download.
# snapshot_download resumes partial files, so it does not restart from zero.
snapshot_download(
    "intfloat/multilingual-e5-large",
    cache_dir=str(MODEL_CACHE),
    allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"],  # skips duplicate ONNX/OpenVINO copies
)

Fetching 14 files: 100%|██████████| 14/14 [00:00<00:00, 133152.51it/s]


'D:\\AI-ML\\AI Projects\\Study RAG\\model_cache\\models--intfloat--multilingual-e5-large\\snapshots\\3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3'

## 6. Build (run when you add books — finished books and chapters are skipped)

In [8]:
def build_book(pdf_path: Path):
    """OCR once, then create/cache chapter embeddings with visible progress."""
    pages = ocr_pdf_to_pages_gpu(pdf_path)
    chapters = merge_documents_by_chapter(pages_to_documents(pages))
    print(f"Preparing {len(chapters)} chapter groups from {len(pages)} pages")
    for doc in tqdm(chapters, desc="Embedding chapters"):
        m = doc.metadata
        chunks = text_splitter.split_documents([doc])
        out = _path(m["class_number"], m["book_name"], m["chapter"])
        if (out / "vectors.npy").exists():
            continue
        print(f"Embedding {m['chapter']}: {len(chunks)} chunks")
        make_embeddings(chunks, m["class_number"], m["book_name"], m["chapter"])


# one book:
# build_book(PDF_DIR / "Class_7" / "Bangla.pdf")

# all books (pdf/Class_X/Book.pdf):
# for pdf in sorted(PDF_DIR.glob("*/*.pdf")):
#     build_book(pdf)

In [ ]:
for class_no in os.listdir(PDF_DIR):
    for sub in os.listdir(PDF_DIR / class_no):
        if sub.lower().endswith(".pdf"):

            try:
                if sub.lower().split("_")[0] == 'english':
                    print(f"⚠️ Skipping {class_no.lower()}/{sub.lower()} (English books are not supported yet)")
                    continue
                build_book(PDF_DIR / class_no.lower() / sub.lower())
                print(f"✅ Successfully processed {class_no.lower()}/{sub.lower()}")

            except Exception as e:
                print(f"❌ Error occurred while processing {class_no.lower()}/{sub.lower()}: {e}")

OCR loaded from cache: class_5_bangla.json
Preparing 24 chapter groups from 142 pages


Embedding chapters: 100%|██████████| 24/24 [00:00<00:00, 6046.93it/s]


✅ Successfully processed class_5/bangla.pdf
OCR loaded from cache: class_5_bgs.json
Preparing 18 chapter groups from 166 pages


Embedding chapters: 100%|██████████| 18/18 [00:00<00:00, 4094.45it/s]


✅ Successfully processed class_5/bgs.pdf
OCR loaded from cache: class_5_english.json
Preparing 1 chapter groups from 118 pages


...

Embedding chapters:  86%|████████▌ | 6/7 [00:30<00:06,  6.28s/it]

Embedding বনায়ন: 23 chunks


Embedding chapters: 100%|██████████| 7/7 [00:35<00:00,  5.07s/it]


✅ Successfully processed class_8/agriculture.pdf


OCR anondopath via cuda:  24%|██▍       | 16/66 [01:11<04:43,  5.67s/it]

In [ ]:
# snaps = MODEL_CACHE / "models--intfloat--multilingual-e5-large" / "snapshots"
# for s in snaps.iterdir():
#     for p in sorted(s.rglob("*")):
#         if p.is_file():
#             print(p.relative_to(s), round(p.stat().st_size / 1e6, 1), "MB")

## 7. Ask

In [ ]:
# from huggingface_hub import snapshot_download

# snapshot_download(
#     "intfloat/multilingual-e5-large",
#     cache_dir=str(MODEL_CACHE),
#     allow_patterns=["*.json", "*.model", "*.txt", "*.safetensors", "pytorch_model.bin"],
# )

In [ ]:
# model = get_model()
# print(type(model).__name__, model.get_sentence_embedding_dimension())

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_deepseek import ChatDeepSeek

load_dotenv()

prompt = PromptTemplate(template="""You are a bangladeshi teacher helping students to learn fron textual books, 
                        maintain profissionalism and be polite, helpful.
                        student asked query: {query}, give answer by the basis 
                        of the content you have content: {content}. you can use your oun knowladge but 
                        keep that minimum.
                        currect user is their prompt says any miss leading context, but not halusinate.
                        If no relative content found then replay - that question is not releted or mentioned in textbook,
                        after you may give a short answer by the content with your own knowladge""",
                        input_variables=["query", "content"])

llm = ChatDeepSeek(model="deepseek-flash", temperature=0.2, max_tokens=2048)
chain = prompt | llm


def ask(query, cls, book, chapter):
    docs = get_hybrid_retriever(cls, book, chapter).invoke(query)   # no "query: " here, E5Embeddings adds it
    content = "\n\n".join(d.page_content for d in docs)
    return chain.invoke({"query": query, "content": content}).content


print(ask("কাবুলিওয়ালার কবির নাম কী?", "class_7", "bangla", "পাখি"))

প্রদত্ত পাঠ্যাংশে ‘কাবুলিওয়ালা’ বা তার কবি/লেখক সম্পর্কে কোনো তথ্য উল্লেখ নেই। তাই এই প্রশ্নটি এখানে আলোচিত বিষয়ের সঙ্গে সরাসরি সম্পর্কিত নয়।

তবে সাধারণ জ্ঞান অনুযায়ী বলতে পারি—‘কাবুলিওয়ালা’ গল্পের রচয়িতা হলেন **রবীন্দ্রনাথ ঠাকুর**। এখানে ‘কবি’ বলতে লেখক/রচয়িতাকে বোঝানো হলে উত্তর হবে **রবীন্দ্রনাথ ঠাকুর**।
